# Reviewer 3 — Comment 8
## Ligand-side FG20 SMARTS coverage audit

This executed notebook applies the exact 20 FG20 SMARTS patterns to the DAVIS and KIBA ligand structure files available in the revision workspace and reproduces the six-inhibitor panel. **For final submission, rerun with the exact ligand files used in the manuscript experiments.** The reviewer-facing response can report vocabulary occupancy without introducing a separate KIBA-version discussion.

In [1]:
from pathlib import Path
import json, numpy as np
from rdkit import Chem, rdBase

FG_SMARTS=(
('CarboxylicAcid','[CX3](=O)[OX2H1]'),('Ester','[CX3](=O)[OX2][#6]'),('Amide','[NX3][CX3](=O)[#6]'),('Anhydride','[CX3](=O)O[CX3](=O)'),('AcylHalide','[CX3](=O)[Cl,Br,I,F]'),('Aldehyde','[CX3H1](=O)[#6]'),('Ketone','[#6][CX3](=O)[#6]'),('Alcohol','[#6;!a][OX2H]'),('Phenol','c[OX2H]'),('Ether','[OX2]([#6])[#6]'),('Nitrile','[CX2]#N'),('Nitro','[$([NX3](=O)=O),$([NX3+](=O)[O-])]'),('Amine_Primary','[NX3;H2][#6]'),('Amine_Secondary','[NX3;H1]([#6])[#6]'),('Amine_Tertiary','[NX3]([#6])([#6])[#6]'),('Thiol','[#16X2H]'),('Thioether','[#16X2]([#6])[#6]'),('Sulfoxide','[#16X3](=O)([#6])[#6]'),('Sulfone','[#16X4](=O)(=O)([#6])[#6]'),('Aryl','c1ccccc1'))
FG=[(n,Chem.MolFromSmarts(s)) for n,s in FG_SMARTS]
SIX={'5291':'Imatinib','123631':'Gefitinib','176870':'Erlotinib','216239':'Sorafenib','3062316':'Dasatinib','5329102':'Sunitinib'}

def load(path):
    txt=Path(path).read_text().strip()
    if txt.startswith('{'):
        obj=json.loads(txt); return [(str(k),str(v)) for k,v in obj.items()]
    out=[]
    for line in txt.splitlines():
        if line.strip() and not line.startswith('#'):
            a,b=line.split(maxsplit=1); out.append((a,b))
    return out

def audit(records):
    matrix=[]
    for _,smi in records:
        m=Chem.MolFromSmiles(smi)
        if m is not None: matrix.append([int(m.HasSubstructMatch(q)) for _,q in FG])
    M=np.asarray(matrix,dtype=int); counts=M.sum(axis=1); ever=int((M.sum(axis=0)>0).sum())
    return {'n_valid':len(M),'ever_active_patterns':ever,'vocabulary_usage_percent':100*ever/20,'zero_match':int((counts==0).sum()),'mean_active':float(counts.mean()),'median_active':float(np.median(counts)),'never_active':[FG_SMARTS[i][0] for i in range(20) if M[:,i].sum()==0]}

base=Path('/mnt/data/smarts_coverage_audit')
davis=load(base/'davis_ligands_can.json')
kiba=load(base/'KIBA_compound_unique.txt')
print('RDKit',rdBase.rdkitVersion)
print('DAVIS',audit(davis))
print('KIBA working structure file',audit(kiba))

RDKit 2025.09.4
DAVIS {'n_valid': 68, 'ever_active_patterns': 15, 'vocabulary_usage_percent': 75.0, 'zero_match': 0, 'mean_active': 3.8676470588235294, 'median_active': 4.0, 'never_active': ['Anhydride', 'AcylHalide', 'Aldehyde', 'Nitro', 'Thiol']}


KIBA working structure file {'n_valid': 2025, 'ever_active_patterns': 17, 'vocabulary_usage_percent': 85.0, 'zero_match': 8, 'mean_active': 3.571851851851852, 'median_active': 4.0, 'never_active': ['Anhydride', 'AcylHalide', 'Thiol']}


In [2]:
d=dict(davis); union=np.zeros(20,dtype=int); panel=[]
for cid,name in SIX.items():
    m=Chem.MolFromSmiles(d[cid]); bits=np.array([int(m.HasSubstructMatch(q)) for _,q in FG]); union|=bits
    panel.append((name,int(bits.sum()),[FG_SMARTS[i][0] for i,b in enumerate(bits) if b]))
for row in panel: print(row)
print('Mean active patterns:',np.mean([x[1] for x in panel]))
print('Union:',int(union.sum()),'/20 =',100*union.sum()/20,'%')

('Imatinib', 4, ['Amide', 'Amine_Secondary', 'Amine_Tertiary', 'Aryl'])
('Gefitinib', 4, ['Ether', 'Amine_Secondary', 'Amine_Tertiary', 'Aryl'])
('Erlotinib', 3, ['Ether', 'Amine_Secondary', 'Aryl'])
('Sorafenib', 4, ['Amide', 'Ether', 'Amine_Secondary', 'Aryl'])
('Dasatinib', 6, ['Amide', 'Alcohol', 'Amine_Secondary', 'Amine_Tertiary', 'Thioether', 'Aryl'])
('Sunitinib', 4, ['Amide', 'Amine_Secondary', 'Amine_Tertiary', 'Aryl'])
Mean active patterns: 4.166666666666667
Union: 7 /20 = 35.0 %
